# LinkedIn IT Job Dataset — Data Cleaning

**Project:** Fit4Job — Skill-Aware Job Matching for Sri Lankan IT Professionals  
**Dataset:** LinkedIn IT Job Postings (Sri Lanka)

This notebook loads the raw scraped dataset, gives a quick overview, identifies all problems, and cleans them step by step.  
It is **reusable** — just change `INPUT_FILE` at the top and re-run all cells.

---

## Setup
Change `INPUT_FILE` to reuse this notebook on a new dataset with the same structure.

In [1]:
import pandas as pd
import numpy as np
import re
import os

# ── Configure paths here ─────────────────────────────────────────

# Input files to combine
FILE_2025 = "/content/2025.csv"
FILE_2026 = "/content/2026.csv"
COMBINED_INPUT_FILE = "/content/combined_raw.csv"

# Combine the input files
print(f"Combining {FILE_2025} and {FILE_2026}...")
df_2025 = pd.read_csv(FILE_2025)
df_2026 = pd.read_csv(FILE_2026)
df_combined = pd.concat([df_2025, df_2026], ignore_index=True)
df_combined.to_csv(COMBINED_INPUT_FILE, index=False)
print(f"Combined data saved to {COMBINED_INPUT_FILE} ({len(df_combined)} rows)")

INPUT_FILE  = COMBINED_INPUT_FILE
OUTPUT_FILE = "cleaned_jobs.csv"
# ─────────────────────────────────────────────────────────────────

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

Combining /content/2025.csv and /content/2026.csv...
Combined data saved to /content/combined_raw.csv (2090 rows)


---
## Load & Overview

In [2]:
df = pd.read_csv(INPUT_FILE)
print(f"Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")
df.head(3)

Rows: 2090  |  Columns: 16


,job_id,title,company,location,posted_date,job_url,search_keyword,scraped_at,description,required_skills,experience_level,employment_type,job_function,industries,job_criteria,num_applicants
0,4314067535,Frontend Developer,qlub,"Colombo, Western Province, Sri Lanka",2025-10-13,https://lk.linkedin.com/jobs/view/frontend-developer-at-qlub-4314067535,software developer,2025-10-14 10:02:28,"Why qlub? Qlub is revolutionizing the dining experience with ultra-fast, sea...","agile, aws, bootstrap, continuous integration, css, devops, docker, git, go,...",Entry level,Full-time,Engineering and Information Technology,Financial Services,Seniority level: Entry level | Employment type: Full-time | Job function: En...,106 applicants
1,4127917718,Full Stack Web Developer – Intern,DoMedia,"Colombo, Western Province, Sri Lanka",2025-01-17,https://lk.linkedin.com/jobs/view/full-stack-web-developer-%E2%80%93-intern-...,software developer,2025-10-14 10:02:30,We are looking for a Full Stack Web Developer who will be responsible for ar...,"cms, css, html, java, javascript, mysql, php, user experience, user interfac...",Internship,Internship,Engineering and Information Technology,Marketing Services,Seniority level: Internship | Employment type: Internship | Job function: En...,Over 200 applicants
2,4301630655,Front End Developer,SenzMate AIoT Lab,"Colombo, Western Province, Sri Lanka",2025-09-17,https://lk.linkedin.com/jobs/view/front-end-developer-at-senzmate-aiot-lab-4...,software developer,2025-10-14 10:02:32,"Description Develop systems by studying operations; designing, developing an...","angularjs, api, backbone, css, cv, react, rest, rest api",Entry level,Full-time,Engineering and Information Technology,IT Services and IT Consulting,Seniority level: Entry level | Employment type: Full-time | Job function: En...,Over 200 applicants


In [3]:
# Data types and null counts at a glance
overview = pd.DataFrame({
    "dtype"     : df.dtypes,
    "null_count": df.isnull().sum(),
    "null_%"    : (df.isnull().mean() * 100).round(1),
    "unique"    : df.nunique(),
})
print(overview)

                   dtype  null_count  null_%  unique
job_id             int64           0     0.0    2046
title             object           0     0.0    1698
company           object           0     0.0     549
location          object           0     0.0      91
posted_date       object           0     0.0     198
job_url           object           0     0.0    2049
search_keyword    object           0     0.0     174
scraped_at        object           0     0.0    2090
description       object           0     0.0    1975
required_skills   object         140     6.7    1608
experience_level  object           0     0.0       7
employment_type   object           0     0.0       6
job_function      object          14     0.7     184
industries        object           1     0.0     266
job_criteria      object           0     0.0     897
num_applicants    object           1     0.0     174


In [4]:
# Key categorical distributions
for col in ["experience_level", "employment_type", "job_function"]:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

print("\nTop 10 industries:")
print(df["industries"].value_counts().head(10).to_string())


experience_level:
experience_level
Mid-Senior level    1083
Entry level          422
Not Applicable       240
Associate            177
Executive             74
Internship            63
Director              31

employment_type:
employment_type
Full-time     1882
Contract       123
Internship      36
Part-time       25
Other           14
Temporary       10

job_function:
job_function
Engineering and Information Technology                                               532
Information Technology                                                               477
Other                                                                                113
Engineering                                                                           80
Design, Art/Creative, and Information Technology                                      64
Sales and Business Development                                                        56
Management and Manufacturing                                                   

---
## Step 2 — Remove Duplicate Postings
Same job scraped under multiple search keywords → keep the first occurrence by `job_url`.

In [5]:
before = len(df)
df = df.drop_duplicates(subset="job_url", keep="first").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows  →  Remaining: {len(df)}")

Removed 41 duplicate rows  →  Remaining: 2049


---
## Step 3 — Remove Non-IT Job Postings

**Strategy — Two signals combined:**

| Signal | Description |
|--------|-------------|
| **IT Industry** | Does the company operate in an IT-related industry? |
| **IT Title** | Does the job title belong to a known IT role? |
| **Non-IT Title** | Does the title clearly belong to a non-IT role (Finance, Sales, HR…)? |

**Decision rule:**
```
KEEP   → IT title  AND  NOT a clear non-IT title
KEEP   → IT industry  AND  NOT a clear non-IT title  (catches: BA, PM, Project Coordinator @ IT firm)
REMOVE → clear non-IT title  (Finance Manager, Sales Manager, etc. regardless of company)
REMOVE → neither IT title  NOR  IT industry
```

**To extend for future datasets:** just add keywords to the lists below — no logic changes needed.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — extend these lists for future datasets
# ─────────────────────────────────────────────────────────────────────────────

# Industries that indicate an IT-sector company
IT_INDUSTRY_KEYWORDS = [
    "it services",
    "software development",
    "software",
    "information technology",
    "technology, information",
    "technology and internet",
    "information and internet",
    "telecommunications",
    "information services",
    "it system",
    "e-learning",
    "research services",
    "computer hardware",
    "desktop computing",
    "data and analytics",
    "internet",
]

# Title substrings that indicate an IT role
IT_TITLE_KEYWORDS = [
    # Core engineering
    "software", "developer", "engineer", "programmer", "coder",
    "frontend", "backend", "fullstack", "full stack", "full-stack",
    "mobile", "ios", "android", "embedded", "firmware",
    # Data & AI
    "data scientist", "data analyst", "data engineer", "data processing",
    "machine learning", "deep learning", "nlp", "computer vision",
    "ai engineer", "ml engineer", "business intelligence", "bi ",
    "analytics",
    # DevOps, Cloud, Infrastructure
    "devops", "devsecops", "sre", "cloud", "infrastructure",
    "network engineer", "network admin", "network analytics", "network audit",
    "systems admin", "sysadmin", "system administrator",
    "it admin", "it support", "intune", "o365", "oracle", "database", "dba",
    # QA & Testing
    "qa ", "quality assurance", "quality engineer",
    "test engineer", "test lead", "test development", "sdet", "automation engineer",
    # Security
    "security", "cybersecurity", "cyber security",
    "penetration", "pen test", "soc ", "governance", "grc",
    "iam ", "identity", "compliance specialist", "risk specialist",
    # Architecture & Design
    "architect", "solution architect", "technical architect",
    "ui/ux", "ux ", "ui ", "ux/ui", "product designer",
    "graphic designer", "visual designer", "web design",
    # Analysis & Consulting
    "business analyst", "technical analyst", "systems analyst",
    "functional analyst", "operations analyst",
    "technical business analyst",
    "application consultant", "integration consultant",
    "implementation", "pre sales", "presales",
    # Project & Delivery (kept alongside IT-industry check)
    "project manager", "project coordinator", "project lead",
    "program manager", "scrum master", "agile",
    "delivery manager", "delivery lead",
    # IT Management
    "technical lead", "tech lead", "engineering manager",
    "it manager", "cto", "ciso",
    "product manager", "product owner",
    # ERP / SaaS platforms
    "sap", "erp", "crm", "salesforce", "dynamics", "hubspot",
    "shopify", "workday", "servicenow",
    # Language/framework-named roles
    "c#", ".net", "java ", "python", "php", "golang", "ruby",
    "react", "angular", "node", "vue", "flutter", "kotlin",
    # Other IT roles
    "helpdesk", "help desk", "service desk", "itsm",
    "intern", "graduate", "researcher",
    "data center",
]

# Title patterns that are unambiguously NON-IT
# Use regex patterns (anchored where useful) for precision
NON_IT_TITLE_PATTERNS = [
    # Finance & Accounting
    r"finance manager", r"financial manager", r"finance controller",
    r"cost controller", r"accounts manager", r"accounting",
    r"payroll", r"actuarial", r"taxation", r"audit manager",
    r"financial operations", r"financial reporting", r"financial analyst",
    # Sales & Marketing (non-technical)
    r"marketing manager", r"sales manager", r"sales executive",
    r"brand manager", r"key account manager", r"account manager",
    r"business development manager", r"business development lead",
    r"cluster sales", r"branch sales", r"area manager",
    r"after sales",
    r"marketing executive", r"social media manager", r"social content manager",
    r"content manager", r"audience planning", r"performance marketing",
    r"digital planning", r"field marketing",
    # HR & Admin
    r"hr manager", r"human resources manager", r"senior hr",
    r"talent acquisition", r"early careers", r"payroll operating",
    # Executive / C-suite (non-IT)
    r"chief executive officer",
    r"managing director",
    r"director of human",
    # Legal & Compliance (non-IT)
    r"legal manager", r"compliance manager", r"regulatory reporting",
    r"corporate affairs",
    # Operations (non-IT)
    r"facilities manager", r"operations manager",
    r"shipping", r"logistics",
    r"head of aftermarket", r"head of operations",
    # Other clearly unrelated
    r"wealth planner", r"relationship manager",
    r"quantity surveyor", r"inspector supervisor",
    r"tour leader",
    r"nurse director", r"medical secretary",
    r"sr manager, eh",
]

# Compile once for speed
_NON_IT_REGEX = re.compile("|".join(NON_IT_TITLE_PATTERNS), re.IGNORECASE)


# ── Helper functions ──────────────────────────────────────────────
def _has_it_industry(industries_str):
    s = str(industries_str).lower()
    return any(kw in s for kw in IT_INDUSTRY_KEYWORDS)

def _has_it_title(title_str):
    t = str(title_str).lower()
    return any(kw in t for kw in IT_TITLE_KEYWORDS)

def _is_clear_non_it(title_str):
    return bool(_NON_IT_REGEX.search(str(title_str)))


# ── Decision function ─────────────────────────────────────────────
def is_it_job(row):
    it_industry = _has_it_industry(row.get("industries", ""))
    it_title    = _has_it_title(row.get("title", ""))
    non_it      = _is_clear_non_it(row.get("title", ""))

    # IT title keyword wins even if a non-IT pattern also matched
    # e.g. 'Business Development Manager - Data Center' → data center is IT
    if it_title:
        return True
    if non_it:
        return False                        # clear non-IT, no IT title → remove
    if it_industry:
        return True                         # IT company + ambiguous title → keep
    return False


before = len(df)
df = df[df.apply(is_it_job, axis=1)].reset_index(drop=True)
print(f"Removed {before - len(df)} non-IT job postings  →  Remaining: {len(df)}")

Removed 326 non-IT job postings  →  Remaining: 1723


In [7]:
# Sanity check — sample of kept titles
print("Sample of kept titles:")
for t in df["title"].sample(min(15, len(df)), random_state=42).tolist():
    print(" ", t)

Sample of kept titles:
  Assistant Manager - Maintenance Engineering
  Software Engineer
  Senior Projects Manager
  Deputy Manager - Internal Audit
  Senior Robotics Engineer
  C/C++ Consultant (Remote)
  Web UI Developer
  Senior Program Manager - Insurance application
  Associate Software Quality Engineer
  Senior Software Engineer
  Manager, ERP Implementations (IGT1)
  Flight Software Engineer (All Levels)
  Associate Quality Engineering Lead
  G1/7 CS2 C - Graphic Designer (Paid Ads Focus)
  SAP FI/CO Consultant


In [8]:
# Sanity check — sample of removed titles (verify no IT roles were dropped)
df_raw = pd.read_csv(INPUT_FILE)
removed = df_raw[~df_raw["job_url"].isin(df["job_url"])]
print(f"Total removed: {len(removed)}")
print("\nSample of removed jobs (confirm these are truly non-IT):")
print(removed[["title", "company", "job_function", "industries"]]
      .sample(min(10, len(removed)), random_state=1)
      .to_string())

Total removed: 329

Sample of removed jobs (confirm these are truly non-IT):
                                                            title                                           company                                       job_function                                            industries
625                              Operations and Logistics Officer  World University Service of Canada (WUSC - EUMC)      Other, Information Technology, and Management                   International Trade and Development
1617                                         Branch Sales Manager                Dynamic Green Plantation (Pvt) Ltd                     Sales and Business Development                                 Investment Management
1626  Tour Leader in Sri lanka for Karmaventura (German speaking)                                    Ventura TRAVEL                       Management and Manufacturing                             Leisure, Travel & Tourism
1661                                   

---
## Step 4 — Remove Closed / Expired Listings

In [9]:
before = len(df)
closed_mask = df["title"].str.contains(r"\[CLOSED\]|\bCLOSED\b", case=False, na=False, regex=True)
print(f"Closed listings found: {closed_mask.sum()}")
if closed_mask.sum():
    print(df[closed_mask][["title", "company"]].to_string())
df = df[~closed_mask].reset_index(drop=True)
print(f"Removed {before - len(df)} closed listings  →  Remaining: {len(df)}")

Closed listings found: 1
                                  title                   company
816  [Closed] Deskside Support Engineer  Balanita Private Limited
Removed 1 closed listings  →  Remaining: 1722


---
## Step 5 — Clean Text Columns

Strip whitespace and remove numeric/code prefixes in titles  
(e.g. `238791 Network Engineer`, `G1/7 CS2 C - Graphic Designer`).

In [10]:
TEXT_COLS = [
    "title", "company", "location",
    "experience_level", "employment_type", "job_function", "industries"
]
for col in TEXT_COLS:
    df[col] = df[col].astype(str).str.strip()

# Remove code prefixes like "G1/7 CS2 C - " or "238791 "
df["title"] = df["title"].str.replace(r"^[A-Z0-9/\s]+-\s+", "", regex=True).str.strip()
df["title"] = df["title"].str.replace(r"^\d{4,}\s+", "", regex=True).str.strip()

print("Text columns cleaned.")
print("\nSample titles after cleaning:")
for t in df["title"].sample(10, random_state=7).tolist():
    print(" ", t)

Text columns cleaned.

Sample titles after cleaning:
  Robotics and Industrial Automation Intern
  Full Stack Web Developer
  Data Engineer
  Principal Software Engineer - C#, .NET
  Senior Engineer – Integration Developer (N8N)
  Senior Executive - Finance
  Attestations and Client Audit Analyst D & A (DORA)
  Software Engineer / Embedded / C++ - Orlando, FL
  Software Engineer, II
  Design Engineer - Fire & Plumbing - 0107


---
## Step 6 — Fix `num_applicants` (text → numeric)

In [11]:
def parse_applicants(val):
    """
    Converts strings like 'Over 200 applicants', 'Be among the first 25 applicants'
    to integers. Returns NaN if no number is found.
    """
    if pd.isna(val):
        return np.nan
    numbers = re.findall(r"\d+", str(val))
    return int(numbers[-1]) if numbers else np.nan

df["num_applicants"] = df["num_applicants"].apply(parse_applicants)

print("num_applicants after conversion:")
print(df["num_applicants"].describe())

num_applicants after conversion:
count    1718.000000
mean       78.498254
std        66.180431
min        25.000000
25%        25.000000
50%        44.000000
75%       121.000000
max       200.000000
Name: num_applicants, dtype: float64


---
## Step 7 — Fix Date Columns (string → datetime)

In [12]:
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")
df["scraped_at"]  = pd.to_datetime(df["scraped_at"],  errors="coerce")

print("posted_date range:", df["posted_date"].min().date(), "→", df["posted_date"].max().date())
print("scraped_at  range:", df["scraped_at"].min().date(),  "→", df["scraped_at"].max().date())
print("\nDate parse failures (NaT):")
print(f"  posted_date: {df['posted_date'].isna().sum()}")
print(f"  scraped_at:  {df['scraped_at'].isna().sum()}")

posted_date range: 2024-10-03 → 2026-02-19
scraped_at  range: 2025-10-14 → 2026-02-19

Date parse failures (NaT):
  posted_date: 0
  scraped_at:  0


---
## Step 8 — Handle Missing Values

In [13]:
# required_skills: no skills listed → empty string
df["required_skills"] = df["required_skills"].fillna("")

# job_function: fill nulls with 'Unknown'
df["job_function"] = (
    df["job_function"]
    .replace("nan", np.nan)
    .fillna("Unknown")
)

# num_applicants: keep as NaN — unknown count is meaningful

print("Remaining nulls after fill:")
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
print(remaining.to_string() if len(remaining) else "  None — all handled ✓")

Remaining nulls after fill:
num_applicants    4


---
## Step 9 — Fix `experience_level` Mislabels

LinkedIn's `Executive` level is intended for C-suite roles. Some senior engineering roles were tagged this way by the scraper. We remap based on the job title.

In [14]:
# Technical role markers — if any appear in title, role is NOT C-suite
TECHNICAL_TITLE_MARKERS = [
    "engineer", "developer", "architect", "analyst", "scientist",
    "lead", "consultant", "programmer", "devops", "qa", "sdet",
    "specialist", "administrator", "tester", "designer",
]

def fix_exec_level(row):
    """Remap 'Executive' → 'Mid-Senior level' for technical roles."""
    if row["experience_level"] == "Executive":
        if any(kw in str(row["title"]).lower() for kw in TECHNICAL_TITLE_MARKERS):
            return "Mid-Senior level"
    return row["experience_level"]

df["experience_level"] = df.apply(fix_exec_level, axis=1)

print("experience_level distribution after fix:")
print(df["experience_level"].value_counts().to_string())

experience_level distribution after fix:
experience_level
Mid-Senior level    935
Entry level         341
Not Applicable      204
Associate           151
Internship           56
Director             23
Executive            12


---
## Step 10 — Drop Redundant Column
`job_criteria` is a pipe-joined duplicate of `experience_level`, `employment_type`, `job_function`, and `industries`.

In [15]:
# Safe to reuse — only drops columns that actually exist
cols_to_drop = [c for c in ["job_criteria"] if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns : {cols_to_drop}")
print(f"Final columns ({df.shape[1]}): {df.columns.tolist()}")

Dropped columns : ['job_criteria']
Final columns (15): ['job_id', 'title', 'company', 'location', 'posted_date', 'job_url', 'search_keyword', 'scraped_at', 'description', 'required_skills', 'experience_level', 'employment_type', 'job_function', 'industries', 'num_applicants']


---
## Step 11 — Final Summary

In [16]:
df_raw = pd.read_csv(INPUT_FILE)

print("=" * 55)
print(" CLEANED DATASET SUMMARY")
print("=" * 55)
print(f"  Original rows   : {len(df_raw)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(df_raw) - len(df)}")
print(f"  Columns         : {df.shape[1]}")
print(f"  Remaining nulls : {df.isnull().sum().sum()}")
print(f"  Date range      : {df['posted_date'].min().date()} → {df['posted_date'].max().date()}")
print()
print("  Experience level:")
for k, v in df["experience_level"].value_counts().items():
    print(f"    {k:<25} {v}")
print()
print("  Top 5 industries:")
for k, v in df["industries"].value_counts().head(5).items():
    print(f"    {str(k)[:55]:<55} {v}")
print("=" * 55)

 CLEANED DATASET SUMMARY
  Original rows   : 2090
  Cleaned rows    : 1722
  Rows removed    : 368
  Columns         : 15
  Remaining nulls : 4
  Date range      : 2024-10-03 → 2026-02-19

  Experience level:
    Mid-Senior level          935
    Entry level               341
    Not Applicable            204
    Associate                 151
    Internship                56
    Director                  23
    Executive                 12

  Top 5 industries:
    IT Services and IT Consulting                           633
    Software Development                                    211
    IT Services and IT Consulting and Financial Services    111
    Software Development and IT Services and IT Consulting  60
    Information Technology & Services                       55


In [17]:
df.head(5)

,job_id,title,company,location,posted_date,job_url,search_keyword,scraped_at,description,required_skills,experience_level,employment_type,job_function,industries,num_applicants
0,4314067535,Frontend Developer,qlub,"Colombo, Western Province, Sri Lanka",2025-10-13,https://lk.linkedin.com/jobs/view/frontend-developer-at-qlub-4314067535,software developer,2025-10-14 10:02:28,"Why qlub? Qlub is revolutionizing the dining experience with ultra-fast, sea...","agile, aws, bootstrap, continuous integration, css, devops, docker, git, go,...",Entry level,Full-time,Engineering and Information Technology,Financial Services,106.0
1,4127917718,Full Stack Web Developer – Intern,DoMedia,"Colombo, Western Province, Sri Lanka",2025-01-17,https://lk.linkedin.com/jobs/view/full-stack-web-developer-%E2%80%93-intern-...,software developer,2025-10-14 10:02:30,We are looking for a Full Stack Web Developer who will be responsible for ar...,"cms, css, html, java, javascript, mysql, php, user experience, user interfac...",Internship,Internship,Engineering and Information Technology,Marketing Services,200.0
2,4301630655,Front End Developer,SenzMate AIoT Lab,"Colombo, Western Province, Sri Lanka",2025-09-17,https://lk.linkedin.com/jobs/view/front-end-developer-at-senzmate-aiot-lab-4...,software developer,2025-10-14 10:02:32,"Description Develop systems by studying operations; designing, developing an...","angularjs, api, backbone, css, cv, react, rest, rest api",Entry level,Full-time,Engineering and Information Technology,IT Services and IT Consulting,200.0
3,4298709681,SOFTWARE DEVELOPER,Ontash,"Colombo, Western Province, Sri Lanka",2025-09-10,https://lk.linkedin.com/jobs/view/software-developer-at-ontash-4298709681,software developer,2025-10-14 10:02:35,Functions: ODOO Developer/ Programmer Must have Bachelor’s degree in Compute...,"agile, bash, c, c#, c++, cordova, f#, git, java, jenkins, jira, linux, mysql...",Entry level,Full-time,Engineering and Information Technology,IT Services and IT Consulting,200.0
4,4310762896,Associate Frontend Developer,Hype Invention,"Colombo, Western Province, Sri Lanka",2025-10-10,https://lk.linkedin.com/jobs/view/associate-frontend-developer-at-hype-inven...,software developer,2025-10-14 10:02:36,About the Role: We are looking for a passionate Associate Frontend Developer...,"css, css3, git, html5, javascript, json, material ui, next.js, react, react....",Entry level,Full-time,Engineering and Information Technology,Advertising Services,200.0


---
## Step 12 — Save Cleaned Dataset

In [18]:
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}  ({len(df)} rows, {df.shape[1]} columns)")

Saved → cleaned_jobs.csv  (1722 rows, 15 columns)
